# Homework 5: Evaluate Your Submission Like the Grader

This notebook runs the grader's exact pipeline on a model you uploaded, using any question CSV in the correct format. By default, this notebook evaluates on `hw5_sample_eval.csv`, a small file with 20 sample questions.

The grader and this notebook call the same functions in `hw5_utils.py`, with the same package versions:

| Step | Function | What it does |
|------|----------|--------------|
| 1 | `load_reference` | Loads the base model's tensor shapes, config, tokenizer, chat template, and generation config at a pinned revision |
| 2 | `check_checkpoint` | Reads only repository metadata, `config.json`, and the safetensors header, and rejects anything that is not trained from `Qwen/Qwen2.5-0.5B-Instruct` |
| 3 | `download_checkpoint` | Downloads `config.json` and `model.safetensors` at the exact commit SHA |
| 4 | `score_checkpoint` | Loads the weights in float32, generates greedily (128 new tokens) with the base model's tokenizer and chat template, and parses answers with `parse_choice_from_boxed` |

## Environment Setup

A GPU runtime is recommended but not required: evaluating 20 questions takes well under a minute on a T4, and a few minutes on a CPU. Run the cell below before any other cell.

In [ ]:
import os
import sys

IS_COLAB = "google.colab" in sys.modules
IS_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ

if IS_COLAB:
    print("Colab environment. Make sure the HW_DIR below matches what you used in the startup notebook.")
    print("If you are asked to restart the runtime, you do not need to do so.")
    print("If you decide to restart the runtime, you will need to re-run this cell.")
    from google.colab import drive
    drive.mount("/content/drive")
    HW_DIR = "/content/drive/MyDrive/CS189/hw/hw5"
    os.chdir(HW_DIR)
    %pip install -q -r colab_requirements.txt
elif IS_DATABRICKS:
    print("Databricks environment. Make sure your base environment is AI v6.")
    %pip install -q -r databricks_requirements.txt
else:
    print("Local development. Make sure your kernel has everything in requirements.txt.")

In [ ]:
from getpass import getpass

import pandas as pd
from huggingface_hub import HfApi

import hw5_utils

---
## What to Evaluate

Enter exactly what you will submit to the [HW5 Model Submission Form](https://forms.gle/HAoRwfytwQEdQHMg8): the repository URL and the full 40-character commit SHA printed by the upload cell. When asked, paste the **fine-grained read-only** token you will submit, not your write token.

`TEST_CSV_PATH` can be any CSV with the columns `id, question, A, B, C, D, E, answer` (`hw5_utils.MCQ_COLUMNS`). You should build your own test set as the sample we provide is too small to draw reliable conclusions.

In [ ]:
REPO_URL = "https://huggingface.co/your-username/cs189-hw5"  # TODO
COMMIT_SHA = "0000000000000000000000000000000000000000"  # TODO: the full commit SHA you will submit
TEST_CSV_PATH = "hw5_sample_eval.csv"

READ_TOKEN = getpass("Paste the fine-grained read-only token you will submit: ").strip()

The grader only needs to read your repository. This cell warns if the token has general access.

In [ ]:
role = (HfApi(token=READ_TOKEN).whoami().get("auth") or {}).get("accessToken", {}).get("role")
if role == "fineGrained":
    print("Token is fine-grained. Make sure it only has read access to your HW5 repository.")
else:
    print(f"WARNING: this token's type is {role!r}. Create a fine-grained token with read access to your HW5 repository only.")

---
## Run the Grader's Pipeline

In [ ]:
questions = hw5_utils.load_mcq_csv(TEST_CSV_PATH)
print(f"Loaded {len(questions)} questions from {TEST_CSV_PATH}")

result = hw5_utils.evaluate_submission(REPO_URL, COMMIT_SHA, READ_TOKEN, questions)

print()
print("Status:", result["status"])
if result.get("detail"):
    print("Detail:", result["detail"])
if result["status"] == "scored":
    print(f"Accuracy: {result['accuracy']:.3f} ({result['num_correct']}/{result['num_questions']})")

- **scored**: the grader will accept this commit and score it the same way on the hidden test set.
- **rejected**: the grader will reject this commit; `Detail` says why. Fix the model, upload again, and evaluate the new commit SHA.
- **error**: the repository could not be read (wrong URL, SHA, or token) or something failed while running. Fix it before you submit; the grader would hit the same problem.

In [ ]:
if result["status"] == "scored":
    details = pd.DataFrame(questions)[["id", "question", "answer"]].merge(
        pd.DataFrame(result["predictions"]), on="id"
    )
    display(details)